# Description
This file contains the code and explanations for the following concepts in machine learning:
1. [Linear Regression](##-Linear-Regression)
- Regularisation
- Regression Metrics
- Polynomial Features
- Overfitting and underfitting

2. [EDA](##-EDA)
- Loading the data (eg `pd.read_csv(path/to/file)`)
- `df.head()`, `df.tail()`, `df.sample()`
- `df.describe()`, `df.info()`, `df.columns`, `df[column].value_counts()`
- `df.shape`, `df.size`,  `df.isnull().sum()`
- and many others...

3. [Data Preprocessing](##-Data-Preprocessing)
- Handling Missing Data
- Feature Scaling - StandardScaler & MinMax Scaler
- Encoding Categorical Data
- ColumnTransformer
- Feature Engineering - Creating better features
- Outlier Detection
- Handling Skew

4. [Cross-Validation](##-Cross-Validation)
- KFold vs StratifiedKFold
- Hyperparameter Tuning - GridSearchCV & RandomizedSearchCV
- Feature Selection


## Imports
Code Cell Below has all the imports that will be used in the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

from sklearn.model_selection import train_test_split, StratifiedGroupKFold, KFold, cross_val_score
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler, RobustScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.compose import ColumnTransformer

np.random.seed(42)

## Linear Regression

### Regularisation Comparison
Compare what each of the regularisation methods do to the coefs. 

In [ ]:
lr = LinearRegression()
lasso = Lasso(alpha=0.1)
ridges = Ridge(alpha=0.1)
eln = ElasticNet(alpha=0.1)

X = np.random.rand(100, 5)
y = 5*X[:,0] + 3*X[:,1] + np.random.randn(100) * 0.1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr.fit(X_train, y_train) # No regularisation
print(f"Linear Coefs: {lr.coef_}") # Assigns useless coefficients to random data. 
lasso.fit(X_train, y_train)
print(f"Lasso Coeffs: {lasso.coef_}") # Identifies useful features and zeros out the rest.
ridges.fit(X_train, y_train)
print(f"Ridge Coeffs: {ridges.coef_}") 
eln.fit(X_train, y_train)
print(f"ElasticNet Coeffs: {eln.coef_}")


### Regression Metrics

In [ ]:
y_pred = lr.predict(X_test) # y_test -> y_pred
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")
print(f"Root Mean Squared Error: {root_mean_squared_error(y_test, y_pred)}")


### PolynomialFeatures, Overfitting & Underfitting

In [ ]:
pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=10)),
    ('lr', LinearRegression())
])
pipeline.fit(X_train.reshape(-1, 1), y_train)

y_pred = pipeline.predict(X_test.reshape(-1, 1))
y_train_pred = pipeline.predict(X_train.reshape(-1, 1))
print("Y Test Pred:", y_pred.round(2))
print("Y train pred", y_train_pred)
print("\nTraining R2 Score", round(r2_score(y_train, y_train_pred), 4))
print("Testing R2 Score:", round(r2_score(y_test, y_pred), 4))

### Varying Degree Parameter 
- in `PolynomialFeatures`, then plot

In [ ]:
degrees = [i for i in range(1, 21)]
trains, tests = list(), list()
for degree in degrees:
    pf = PolynomialFeatures(degree=degree)
    # Transforming with fit_transform()
    X_train_poly = pf.fit_transform(X_train.reshape(-1, 1))
    X_test_poly = pf.fit_transform(X_test.reshape(-1, 1))
    # Using Linear Regression.
    newregr = LinearRegression()
    newregr.fit(X_train_poly, y_train)
    y_train_pred = newregr.predict(X_train_poly)
    y_test_pred = newregr.predict(X_test_poly)
    # Mean Absolute Errors, for training and tests. 
    train_err = mean_squared_error(y_train, y_train_pred)
    test_err = mean_squared_error(y_test, y_test_pred)
    # Data appended to list for graphing. 
    trains.append(train_err)
    tests.append(test_err)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(degrees, trains, label='Training Error', marker='o')
plt.plot(degrees, tests, label='Testing Error', marker='x')
plt.title('Learning Curve')
plt.xlabel('Degrees')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.grid(True)
plt.show()

## EDA

In [ ]:
# Load the data eg from a csv file
path = "../Data/athelete.parquet"
df = pd.read_parquet(path) # Parquet file type takes lower memmory than csv for same data. 

df.head() # first 5 rows or df.head(3) for first 3 rows. 

In [ ]:
df.sample() # A random row, or df.sample(5) for 5 random rows.

In [ ]:
df.describe()

In [ ]:
df.isnull().sum() # Null counts

In [ ]:
df.shape

## Data Preprocessing

### Handling Missing Data

In [ ]:
imp_mean = SimpleImputer(strategy='mean')
imp_mode = SimpleImputer(strategy='most_frequent')
imp_const = SimpleImputer(strategy='comstant', fill_value=0)

df[['Weight', 'Height']] = imp_mean.fit_transform(df[['Weight', 'Height']])
df[['Age']] = imp_mode.fit_transform(df[['Age']])
df[['Medal']] = imp_const.fit_transform(df[['Medal']])
df.isnull().sum() # No more nulls. 

### Feature Scaling

In [ ]:
stdscaler = StandardScaler()
mmscaler = MinMaxScaler()
rbscaler = RobustScaler()

df[['Age']] = stdscaler.fit_transform(df[['Age']])
df[['Height']] = mmscaler.fit_transform(df[['Height']])
df[['Weight']] = rbscaler.fit_transform(df[['Weight']])
df.head() # Different scaling techniques. 

### Encoding Categorical Data

In [ ]:
ohe = OneHotEncoder() # No Ordinal data so OrdinalEncoder is not used. See temp.ipynb for an example. 




## Cross-Validation